# RAG Red Team Workflow Exercise

Use this notebook to generate two artifacts for the EnterpriseSummarizer assessment:

1. `docs/llm_generated_attack_vectors.md`
2. `docs/red_team_charter.md`

The workflow intentionally uses two LLM requests. The first request generates proposed attacks. The second request appends those attacks to the context bundle and generates a charter from them.

## 1. Load the Exercise Materials

In [ ]:
from pathlib import Path
import json

cwd = Path.cwd()
EXERCISE_ROOT = cwd.parent if cwd.name == "notebooks" else Path("module-3-apply-ai-red-teaming/exercise/solution")

api_spec = json.loads((EXERCISE_ROOT / "data/rag_api_spec.json").read_text(encoding="utf-8"))
sample_requests = json.loads((EXERCISE_ROOT / "data/sample_summary_requests.json").read_text(encoding="utf-8"))
architecture = (EXERCISE_ROOT / "docs/deployment_architecture.md").read_text(encoding="utf-8")
model_card = (EXERCISE_ROOT / "docs/model_card.md").read_text(encoding="utf-8")
infrastructure_notes = (EXERCISE_ROOT / "docs/infrastructure_notes.md").read_text(encoding="utf-8")
retrieval_policy = (EXERCISE_ROOT / "docs/retrieval_access_policy.md").read_text(encoding="utf-8")
structured_prompt = (EXERCISE_ROOT / "prompts/structured_vulnerability_discovery_prompt.md").read_text(encoding="utf-8")

attack_vectors_path = EXERCISE_ROOT / "docs/llm_generated_attack_vectors.md"
charter_path = EXERCISE_ROOT / "docs/red_team_charter.md"
attack_vectors_template = attack_vectors_path.read_text(encoding="utf-8")
charter_template = charter_path.read_text(encoding="utf-8")

print(f"Loaded API: {api_spec['info']['title']}")
print(f"Loaded sample requests: {len(sample_requests)}")

## 2. Complete the Structured Prompt

Open `prompts/structured_vulnerability_discovery_prompt.md` and replace the TODO sections before continuing. Your prompt should ask for prioritized findings, supporting evidence, mitigation ideas, and false-positive notes.

In [ ]:
if "TODO" in structured_prompt:
    raise ValueError("Complete prompts/structured_vulnerability_discovery_prompt.md before running the LLM requests.")

print(structured_prompt[:1500])

## 3. Build the Context Bundle

The context bundle supplies system details plus templates for both artifacts.

In [ ]:
context_bundle = f"""
{structured_prompt}

## API Specification
```json
{json.dumps(api_spec, indent=2)}
```

## Deployment Architecture
{architecture}

## Model Card
{model_card}

## Infrastructure Notes
{infrastructure_notes}

## Retrieval Access Policy
{retrieval_policy}

## Sample Summary Requests
```json
{json.dumps(sample_requests, indent=2)}
```

## Artifact Template: LLM-Generated Attack Vectors
Use this file as the required structure. Replace TODO content with a complete generated artifact.

{attack_vectors_template}

## Artifact Template: Red Team Charter
Use this file as the required structure. The charter must be based on the generated attack vectors from the first request.

{charter_template}
""".strip()

print(context_bundle[:3000])
print("\n--- context bundle truncated for display ---")

## 4. Request 1: Generate and Save Proposed Attacks

Run this cell after configuring your OpenAI API key or adapting the call for your local LLM endpoint.

In [ ]:
import os
from openai import OpenAI

# Paste your OpenAI API key here before running this cell.
# If you're using a Vocareum-issued key (starts with "voc-"), the
# Vocareum base URL is set automatically.
OPENAI_API_KEY = "PASTE_YOUR_API_KEY_HERE"

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if OPENAI_API_KEY.startswith("voc-"):
    os.environ["OPENAI_BASE_URL"] = "https://openai.vocareum.com/v1"
else:
    os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"

client = OpenAI()

attack_vector_request = f"""
{context_bundle}

## Generation Task

Generate the first artifact only: `docs/llm_generated_attack_vectors.md`.

Requirements:
- Follow the attack-vector artifact template.
- Prioritize risks as P1, P2, or P3.
- Include supporting evidence from the provided context.
- Include mitigation ideas and false-positive or validation notes.
- Include rejected or deferred speculative findings.
- Return Markdown only, with no code fence around the artifact.
""".strip()

attack_vector_response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "system",
            "content": [
                {
                    "type": "input_text",
                    "text": "You are a security analysis assistant generating evidence-grounded AI red team planning hypotheses."
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": attack_vector_request,
                }
            ],
        },
    ],
)

generated_attack_vectors = attack_vector_response.output_text.strip() + "\n"
attack_vectors_path.write_text(generated_attack_vectors, encoding="utf-8")
print(f"Saved {attack_vectors_path}")
print(generated_attack_vectors[:2000])

## 5. Request 2: Generate and Save the Charter

This request appends the generated attack vectors to the context bundle and asks for a charter in the provided format.

In [ ]:
charter_context_bundle = f"""
{context_bundle}

## Generated Attack Vectors From First LLM Request

{generated_attack_vectors}
""".strip()

charter_request = f"""
{charter_context_bundle}

## Generation Task

Generate the second artifact only: `docs/red_team_charter.md`.

Requirements:
- Follow the red team charter artifact template.
- Base the Objectives and Success Criteria on the generated attack vectors.
- Include scope, rules of engagement, success criteria, deliverables, and assessment notes.
- Keep the charter appropriate for a limited-deployment internal RAG service.
- Return Markdown only, with no code fence around the artifact.
""".strip()

charter_response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "system",
            "content": [
                {
                    "type": "input_text",
                    "text": "You are a security analysis assistant generating concise, evidence-grounded AI red team planning artifacts."
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": charter_request,
                }
            ],
        },
    ],
)

generated_charter = charter_response.output_text.strip() + "\n"
charter_path.write_text(generated_charter, encoding="utf-8")
print(f"Saved {charter_path}")
print(generated_charter[:2000])

## 6. Review for False Positives and Operational Risk

Before submitting, review both saved artifacts. Remove unsupported claims, downgrade speculative risks, and make sure the mitigations are actionable.

In [ ]:
print("# Saved attack vectors\n")
print(attack_vectors_path.read_text(encoding="utf-8"))
print("\n# Saved charter\n")
print(charter_path.read_text(encoding="utf-8"))